# 09 Model Evaluation — M1-M8 + T12/T18/T24 (Normalized Targets)

**Key design**: Targets are normalized by each instrument's std before training (`Target_{h}d_norm`).
This corrects for scale differences between instruments (equivalent to WLS / heteroskedasticity correction).
Predictions are denormalized (× instr_std) for bps-level interpretation.

1. **Overall Summary** — IC, ICIR, Direction Accuracy (on normalized scale)
2. **IC by Rate Label** — comparison across all 11 instruments
3. **BOJ Swap vs Tenor OIS** — accuracy by rate type
4. **IC by Days to MPM**
5. **Fold IC Stability**
6. **Cross-Sectional P&L** — Top4 Long / Bottom4 Short
7. **Latest Date Predictions** — denormalized to bps

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import spearmanr

from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, calculate_metrics

EXCEL_PATH  = '../data/BOJ_data.xlsx'
MEETING_CSV = '../data/BOJ_meeting_history.csv'
START_DATE  = '2024-01-01'

MI_LABEL    = {1:'M1',2:'M2',3:'M3',4:'M4',5:'M5',6:'M6',7:'M7',8:'M8',
               10:'T12',11:'T18',12:'T24'}
LABEL_ORDER = ['M1','M2','M3','M4','M5','M6','M7','M8','T12','T18','T24']

df_raw    = load_and_clean_data(EXCEL_PATH, MEETING_CSV)
df_feat   = generate_features(df_raw)
df_pooled = pool_boj_data(df_feat)

# Per-instrument std for denormalization
# Rates are in % units (e.g. 0.74625 = 0.74625%), so to convert to bps: * 100
instr_std_3d = df_pooled.groupby('Rate_Label')['Target_3d_std'].first()
instr_std_5d = df_pooled.groupby('Rate_Label')['Target_5d_std'].first()

print(f'Pooled shape : {df_pooled.shape}')
print(f'Date range   : {df_pooled["Date"].min().date()} to {df_pooled["Date"].max().date()}')
print(f'Rate labels  : {sorted(df_pooled["Rate_Label"].unique())}')
print()
print('Instrument std (Target_3d) in bps — rates are in % so std[%] * 100 = std[bps]:')
print((instr_std_3d * 100).round(4).rename('std (bps)').to_frame().T)

In [ ]:
# Train on normalized targets (Target_{h}d_norm) for equal-scale training across instruments
print('Running 3d walk-forward (normalized target)...')
res_3d, model_3d, X_test_3d, y_test_3d, X_train_3d = walk_forward_validation(
    df_pooled, 'Target_3d_norm', START_DATE, return_model=True)

print('Running 5d walk-forward (normalized target)...')
res_5d, model_5d, X_test_5d, y_test_5d, X_train_5d = walk_forward_validation(
    df_pooled, 'Target_5d_norm', START_DATE, return_model=True)

# Add Rate_Label and Days_to_MPM
res_3d['Rate_Label'] = res_3d['Meeting_Index'].map(MI_LABEL)
res_5d['Rate_Label'] = res_5d['Meeting_Index'].map(MI_LABEL)

dtm_map = df_pooled[['Date','Meeting_Index','Days_to_MPM']].drop_duplicates()
res_3d  = res_3d.merge(dtm_map, on=['Date','Meeting_Index'], how='left')
res_5d  = res_5d.merge(dtm_map, on=['Date','Meeting_Index'], how='left')

# Denormalized columns:
#   Actual/Pred are in normalized (sigma) units
#   instr_std is in % units  →  * 100 converts to bps  (NOT * 10000)
for res, std_map in [(res_3d, instr_std_3d), (res_5d, instr_std_5d)]:
    res['instr_std'] = res['Rate_Label'].map(std_map)
    res['Actual_bps'] = res['Actual'] * res['instr_std'] * 100
    res['Pred_bps']   = res['Pred']   * res['instr_std'] * 100

print(f'\nOOS samples — 3d: {len(res_3d)}, 5d: {len(res_5d)}')
print(f'Folds       — 3d: {res_3d["Fold"].nunique()}, 5d: {res_5d["Fold"].nunique()}')
print()
print('Actual_bps sanity check (should be ~single-digit bps for M1):')
print(res_3d[res_3d['Rate_Label']=='M1']['Actual_bps'].describe().round(4))

## 1. Overall Summary

IC is computed on normalized scale (fair cross-instrument comparison).
Sharpe uses sign(Pred_norm) × Actual_norm — consistent with training objective.

In [ ]:
def compute_summary(res, label):
    # IC on normalized values
    m = calculate_metrics(res['Actual'], res['Pred'])
    fold_ics = res.groupby('Fold').apply(lambda g: spearmanr(g['Actual'], g['Pred'])[0])
    icir   = fold_ics.mean() / fold_ics.std() if fold_ics.std() > 0 else np.nan
    pnl    = np.sign(res['Pred']) * res['Actual']  # normalized units
    sharpe = pnl.mean() / pnl.std() * np.sqrt(252) if pnl.std() > 0 else np.nan
    return {
        'Target'              : label,
        'IC (Spearman)'       : round(m['IC'], 4),
        'ICIR'                : round(icir, 4),
        'Dir Acc. (All)'      : f"{m['Direction_Accuracy']:.1%}",
        'Dir Acc. (LargeMove)': f"{m['Direction_Accuracy_LargeMove']:.1%}",
        'Sharpe (sign-follow)': round(sharpe, 3),
        'Folds'               : int(res['Fold'].nunique()),
        'OOS Samples'         : len(res),
    }

df_summary = pd.DataFrame([
    compute_summary(res_3d, '3d'),
    compute_summary(res_5d, '5d'),
])
print('=== Overall Model Performance Summary (normalized scale) ===')
df_summary

## 2. IC by Rate Label

IC computed on normalized scale — each instrument contributes equally regardless of volatility level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('IC by Rate Label — BOJ Swap (M1-M8) vs Tenor OIS (T12/T18/T24)', fontsize=13)

for ax, res, title in [(axes[0], res_3d, 'Target 3d (normalized)'),
                       (axes[1], res_5d, 'Target 5d (normalized)')]:
    ics = res.groupby('Rate_Label').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0])
    ics = ics.reindex([l for l in LABEL_ORDER if l in ics.index])
    colors = ['steelblue' if v > 0 else 'salmon' for v in ics]
    x = np.arange(len(ics))
    ax.bar(x, ics.values, color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(ics.mean(), color='navy', linestyle='--', linewidth=1.2,
               label=f'Mean={ics.mean():.3f}')
    for i, v in enumerate(ics.values):
        ax.text(i, v + 0.008 * (1 if v >= 0 else -1), f'{v:.3f}',
                ha='center', fontsize=8)
    boj_n = sum(1 for l in ics.index if l.startswith('M'))
    ax.axvline(boj_n - 0.5, color='gray', linestyle=':', linewidth=1.5)
    ax.text(boj_n - 0.7, ics.max() * 0.9, 'BOJ Swap',  fontsize=8, color='gray', ha='right')
    ax.text(boj_n + 0.2, ics.max() * 0.9, 'Tenor OIS', fontsize=8, color='gray', ha='left')
    ax.set_xticks(x)
    ax.set_xticklabels(list(ics.index))
    ax.set_title(title)
    ax.set_ylabel('IC (Spearman)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. BOJ Swap vs Tenor OIS — Accuracy by Rate Type

In [ ]:
def type_comparison(res):
    rows = []
    for rate_type, mask in [('BOJ Swap (M1-M8)',   ~res['Rate_Label'].str.startswith('T')),
                             ('Tenor OIS (T12-T24)', res['Rate_Label'].str.startswith('T'))]:
        sub = res[mask]
        if len(sub) == 0:
            continue
        m = calculate_metrics(sub['Actual'], sub['Pred'])
        fold_ics = sub.groupby('Fold').apply(lambda g: spearmanr(g['Actual'], g['Pred'])[0])
        icir = fold_ics.mean() / fold_ics.std() if fold_ics.std() > 0 else np.nan
        rows.append({
            'Type'                : rate_type,
            'IC'                  : round(m['IC'], 4),
            'ICIR'                : round(icir, 4),
            'Dir Acc. (All)'      : f"{m['Direction_Accuracy']:.1%}",
            'Dir Acc. (LargeMove)': f"{m['Direction_Accuracy_LargeMove']:.1%}",
            'N'                   : len(sub),
        })
    return pd.DataFrame(rows)

print('=== 3d Model ===')
display(type_comparison(res_3d))
print('=== 5d Model ===')
display(type_comparison(res_5d))

## 4. IC by Days to MPM

In [ ]:
def dtm_bucket(d):
    if pd.isna(d): return 'Unknown'
    if d <= 5:     return '<=5 (Pre-MPM)'
    if d <= 15:    return '6-15'
    return '16+'

BUCKET_ORDER = ['<=5 (Pre-MPM)', '6-15', '16+']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('IC by Days to Next MPM', fontsize=13)

for ax, res, title in [(axes[0], res_3d, 'Target 3d'), (axes[1], res_5d, 'Target 5d')]:
    r = res.copy()
    r['bucket'] = r['Days_to_MPM'].apply(dtm_bucket)
    bkt_ics = r.groupby('bucket').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0]).reindex(BUCKET_ORDER)
    bkt_n = r.groupby('bucket').size().reindex(BUCKET_ORDER)
    colors = ['steelblue' if v > 0 else 'salmon' for v in bkt_ics]
    ax.bar(BUCKET_ORDER, bkt_ics.values, color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    for j, (v, bkt) in enumerate(zip(bkt_ics.values, BUCKET_ORDER)):
        ax.text(j, v + 0.008 * (1 if v >= 0 else -1),
                f'{v:.3f}\n(n={bkt_n[bkt]})', ha='center', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel('IC (Spearman)')
    ax.set_xlabel('Days to Next MPM')

plt.tight_layout()
plt.show()

## 5. Fold IC Stability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('IC by Fold — Walk-Forward OOS Stability', fontsize=13)

for ax, res, title in [(axes[0], res_3d, 'Target 3d'), (axes[1], res_5d, 'Target 5d')]:
    fold_ics   = res.groupby('Fold').apply(lambda g: spearmanr(g['Actual'], g['Pred'])[0])
    fold_start = res.groupby('Fold')['Date'].min().dt.strftime('%Y-%m')
    colors = ['steelblue' if v > 0 else 'salmon' for v in fold_ics]
    ax.bar(range(len(fold_ics)), fold_ics.values, color=colors,
           edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    icir = fold_ics.mean() / fold_ics.std() if fold_ics.std() > 0 else np.nan
    ax.axhline(fold_ics.mean(), color='navy', linestyle='--', linewidth=1.2,
               label=f'Mean={fold_ics.mean():.3f}  ICIR={icir:.2f}')
    ax.set_xticks(range(len(fold_ics)))
    ax.set_xticklabels(fold_start.values, rotation=45, ha='right', fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('IC (Spearman)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6. Cross-Sectional P&L Simulation

**Strategy**: Each trading day, rank all 11 instruments by predicted return (in normalized units).
Go long top 4, short bottom 4 (middle 3 neutral).

Because predictions are in normalized (z-score) units, the ranking is scale-fair across instruments.
P&L is computed in two ways:
- **Normalized P&L** (sigma units): uses normalized actuals — equal risk contribution per instrument
- **bps P&L**: uses denormalized actuals — reflects actual rate move magnitude

In [ ]:
def cross_sectional_pnl(res, top_n=4, bottom_n=4, actual_col='Actual'):
    """
    Rank by Pred (normalized z-score) each day.
    Long top_n, Short bottom_n.
    actual_col: 'Actual' (normalized sigma units) or 'Actual_bps'
    """
    rows = []
    for date, grp in res.groupby('Date'):
        if len(grp) < top_n + bottom_n:
            continue
        grp = grp.sort_values('Pred', ascending=False).reset_index(drop=True)
        long_ret  = grp.head(top_n)[actual_col].mean()
        short_ret = grp.tail(bottom_n)[actual_col].mean()
        rows.append({
            'Date'        : date,
            'PnL'         : long_ret - short_ret,
            'Long_Labels' : ','.join(grp.head(top_n)['Rate_Label'].values),
            'Short_Labels': ','.join(grp.tail(bottom_n)['Rate_Label'].values),
        })
    return pd.DataFrame(rows).sort_values('Date').reset_index(drop=True)

# Normalized P&L (sigma units) — primary metric
cs_3d_norm = cross_sectional_pnl(res_3d, actual_col='Actual')
cs_5d_norm = cross_sectional_pnl(res_5d, actual_col='Actual')
# bps P&L — for interpretability
cs_3d_bps  = cross_sectional_pnl(res_3d, actual_col='Actual_bps')
cs_5d_bps  = cross_sectional_pnl(res_5d, actual_col='Actual_bps')

print(f'Trading days — 3d: {len(cs_3d_norm)}, 5d: {len(cs_5d_norm)}')

In [ ]:
def plot_cs(ax, cs, title, unit):
    cs = cs.copy()
    cs['CumPnL'] = cs['PnL'].cumsum()
    total  = cs['CumPnL'].iloc[-1]
    sharpe = cs['PnL'].mean() / cs['PnL'].std() * np.sqrt(252) if cs['PnL'].std() > 0 else np.nan
    max_dd = (cs['CumPnL'].cummax() - cs['CumPnL']).max()
    hit    = (cs['PnL'] > 0).mean()
    ax.plot(cs['Date'], cs['CumPnL'], color='steelblue', linewidth=1.3)
    ax.fill_between(cs['Date'], cs['CumPnL'], 0,
                    where=cs['CumPnL'] >= 0, alpha=0.15, color='steelblue')
    ax.fill_between(cs['Date'], cs['CumPnL'], 0,
                    where=cs['CumPnL'] <  0, alpha=0.15, color='salmon')
    ax.axhline(0, color='black', linewidth=0.6)
    stats = f'Total={total:.3f}  Sharpe={sharpe:.2f}  MaxDD={max_dd:.3f}  Hit={hit:.1%}'
    ax.set_title(f'{title}\n{stats}', fontsize=10)
    ax.set_ylabel(f'Cum P&L ({unit})')
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Cross-Sectional P&L — Top4 Long / Bottom4 Short (11 instruments)', fontsize=13)

plot_cs(axes[0][0], cs_3d_norm, '3d Model — Normalized (sigma)', 'sigma')
plot_cs(axes[0][1], cs_3d_bps,  '3d Model — Raw (bps)',          'bps')
plot_cs(axes[1][0], cs_5d_norm, '5d Model — Normalized (sigma)', 'sigma')
plot_cs(axes[1][1], cs_5d_bps,  '5d Model — Raw (bps)',          'bps')

plt.tight_layout()
plt.show()

### 6-2. Long/Short Selection Frequency by Rate Label

In [ ]:
def label_frequency(cs, role):
    col = 'Long_Labels' if role == 'Long' else 'Short_Labels'
    freq = pd.Series(','.join(cs[col].values).split(',')).value_counts()
    return freq.reindex(LABEL_ORDER).fillna(0).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Long/Short Selection Frequency by Rate Label (5d Model)', fontsize=13)

for row, (cs, model_label) in enumerate([(cs_5d_norm, '5d Normalized'), (cs_5d_bps, '5d bps')]):
    for col, role in enumerate(['Long', 'Short']):
        ax = axes[row][col]
        freq  = label_frequency(cs, role)
        color = 'steelblue' if role == 'Long' else 'salmon'
        ax.bar(freq.index, freq.values, color=color, edgecolor='black', linewidth=0.5)
        boj_n = sum(1 for l in freq.index if l.startswith('M'))
        ax.axvline(boj_n - 0.5, color='gray', linestyle=':', linewidth=1.5)
        ax.set_title(f'{model_label} — {role}')
        ax.set_ylabel('Selection Count')
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Latest Date Predictions — All 11 Instruments

Model outputs normalized predictions → denormalized to bps for display.

In [ ]:
feat_cols_3d = model_3d.feature_name()
feat_cols_5d = model_5d.feature_name()

latest_date = df_pooled['Date'].max()
latest_rows = (
    df_pooled[df_pooled['Date'] == latest_date]
    .sort_values('Meeting_Index')
    .copy()
)
latest_rows['Rate_Label'] = latest_rows['Meeting_Index'].map(MI_LABEL)

# Predict (normalized sigma units) then denormalize:
#   pred_bps = pred_norm * instr_std[%] * 100[bps/%]
pred_3d_norm = model_3d.predict(latest_rows[feat_cols_3d])
pred_5d_norm = model_5d.predict(latest_rows[feat_cols_5d])
pred_3d_bps  = pred_3d_norm * latest_rows['Rate_Label'].map(instr_std_3d).values * 100
pred_5d_bps  = pred_5d_norm * latest_rows['Rate_Label'].map(instr_std_5d).values * 100

labels = latest_rows['Rate_Label'].values
x      = np.arange(len(labels))
width  = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars3 = ax.bar(x - width/2, pred_3d_bps, width, label='3d prediction',
               color='steelblue', edgecolor='black', linewidth=0.5)
bars5 = ax.bar(x + width/2, pred_5d_bps, width, label='5d prediction',
               color='coral',     edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
boj_n = sum(1 for l in labels if l.startswith('M'))
ax.axvline(boj_n - 0.5, color='gray', linestyle=':', linewidth=1.5)
for bar in list(bars3) + list(bars5):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + (0.3 if h >= 0 else -0.3),
            f'{h:.2f}', ha='center', va=('bottom' if h >= 0 else 'top'), fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Rate Label')
ax.set_ylabel('Predicted Change (bps)')
ax.set_title(
    f'Latest Date Predictions as of {latest_date.date()}\n'
    f'3-day and 5-day predicted OIS spread change — all 11 instruments'
)
ax.legend()
plt.tight_layout()
plt.show()

# Summary table
df_pred = pd.DataFrame({
    'Rate_Label'    : labels,
    'Type'          : ['BOJ Swap' if l.startswith('M') else 'Tenor OIS' for l in labels],
    'Days_to_MPM'   : latest_rows['Days_to_MPM'].values,
    'Pred_3d_norm'  : pred_3d_norm.round(4),
    'Pred_3d (bps)' : pred_3d_bps.round(2),
    'Pred_5d_norm'  : pred_5d_norm.round(4),
    'Pred_5d (bps)' : pred_5d_bps.round(2),
})
print(f'Latest date: {latest_date.date()}')
df_pred